In [1]:
# chargement des librairies
import glob as g
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
from ydata_profiling import ProfileReport
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# extraction for old data
def extract_old(path):
    # chargement du dataset
    extracted_data = pd.read_csv(path)
    return extracted_data

In [3]:
# transformation for old data
def transform_old(data):
    # conversion de la colonne Date en un object de type datetime
    data['Date'] = pd.to_datetime(data['Date'])
    # création de nouvelles colonnes reliées à la date
    data['Year'] = data['Date'].dt.year
    data['Month'] = data['Date'].dt.month
    data['Day'] = data['Date'].dt.day
    # suppression des lignes dont toutes les valeurs sont nulles à l'exception des valeurs Date et Location
    data.dropna(axis = 0, thresh = 3, inplace = True)
    # suppression de la colonne Date
    data.drop('Date', axis = 1, inplace=True)
    # suppression de la colone RISK_MM
    data.drop(["RISK_MM"], axis=1, inplace=True)
    # transformation des noms de colonne en miniscule
    data.columns = data.columns.str.lower()
    # suppression des lignes ayant des valeurs manquantes dans la variable réponse (raintomorrow)
    data.dropna(subset=['raintomorrow'], inplace = True)
          
    return data

In [4]:
# chargement
def load(target_file, data_to_load):
    data_to_load.to_csv(target_file, index = False)

In [2]:
# executer les opérations ETL sur le fichier csv des données anciennes

# chemin vers le fichier csv à manipuler
input_path = '../../datasets/raw/dec2008-march2023/weatherAUS-2.csv'

# chemin de sauvegarde du fichier final
output_path = '../../datasets/processed/dec2008-march2023/processed-weather-aus-2008-2023.csv'

# execution du pipeline ETL
extracted_data = extract_old(path = input_path)
transformed_data = transform_old(data = extracted_data)
loaded_data = load(target_file = output_path, data_to_load = transformed_data)


NameError: name 'extract_old' is not defined

In [4]:
Previous_Date = datetime.today() - timedelta(days=1)
Previous_Date.day

20

In [10]:
datetime.today() - timedelta(days = 1)

datetime.datetime(2023, 7, 20, 21, 8, 33, 519284)

In [11]:
# selection des données jusqu'à une certaine profondeur (en jours) dans le passé
# à partir d'une date passée exprimée en nombre de jour par rapport au présent
def select_data_ndays_ago(data, ndays = 0, start = 0):
    start_date = datetime.today() - timedelta(days = start)
 
    res = pd.DataFrame(columns = data.columns)
    
    for d in range(0, ndays + 1):
        previous_date = start_date - timedelta(days = d)
        day = previous_date.day
        month = previous_date.month
        year = previous_date.year

        df = data[(data.year== year)&(data.month==month)&(data.day==day)]
        res = pd.concat([res, df], axis=0, ignore_index = True)

    return res

In [12]:
# chemin vers le fichier csv à manipuler
input_path = '../../datasets/processed/latest/processed-weather-aus-latest.csv'

df_1=pd.read_csv(input_path)

df_1

,location,mintemp,maxtemp,rainfall,evaporation,sunshine,windgustdir,windgustspeed,winddir9am,winddir3pm,...,pressure3pm,cloud9am,cloud3pm,temp9am,temp3pm,raintoday,raintomorrow,year,month,day
0,Ballarat,6.5,12.2,2.4,NaN,NaN,SW,44.0,SW,SSW,...,1020.0,7.0,4.0,8.4,11.3,Yes,No,2023,6,1
1,Ballarat,-0.7,11.4,0.0,NaN,NaN,N,28.0,E,N,...,1022.5,8.0,4.0,5.0,9.8,No,Yes,2023,6,2
2,Ballarat,5.0,14.7,3.8,NaN,NaN,ESE,41.0,NNW,,...,1026.1,8.0,8.0,9.0,14.1,Yes,No,2023,6,3
3,Ballarat,6.4,17.4,0.4,NaN,NaN,NNE,48.0,,NE,...,1027.5,NaN,NaN,10.4,16.6,No,No,2023,6,4
4,Ballarat,10.3,14.6,0.2,NaN,NaN,NNE,54.0,NE,NE,...,1021.8,1.0,8.0,12.4,11.8,No,Yes,2023,6,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2396,Perth,2.8,16.4,1.0,2.8,9.3,NE,28.0,E,NE,...,1027.5,0.0,0.0,9.1,15.7,Yes,No,2023,7,15
2397,Perth,1.9,19.0,0.0,2.8,9.4,NNE,33.0,NE,N,...,1022.1,0.0,0.0,10.0,18.4,No,No,2023,7,16
2398,Perth,6.8,20.7,0.0,1.0,4.9,NNE,17.0,NNE,ESE,...,1020.9,7.0,3.0,11.8,19.5,No,No,2023,7,17
2399,Perth,7.6,19.4,0.0,0.8,1.5,NNW,41.0,N,NW,...,1012.1,7.0,8.0,13.6,19.0,No,Yes,2023,7,18


In [17]:
df_2 = select_data_ndays_ago(data = df_1, ndays = 1, start = 2)
df_2

,location,mintemp,maxtemp,rainfall,evaporation,sunshine,windgustdir,windgustspeed,winddir9am,winddir3pm,...,pressure3pm,cloud9am,cloud3pm,temp9am,temp3pm,raintoday,raintomorrow,year,month,day
0,Ballarat,-3.1,NaN,1.6,NaN,NaN,NaN,NaN,N,N,...,1023.0,NaN,7.0,0.9,10.0,Yes,NaN,2023,7,19
1,Wollongong,8.6,NaN,0.0,NaN,NaN,NaN,NaN,W,S,...,1022.5,NaN,NaN,12.1,14.2,No,NaN,2023,7,19
2,Penrith,4.0,NaN,0.2,NaN,NaN,NaN,NaN,,SSE,...,NaN,NaN,NaN,11.0,17.2,No,NaN,2023,7,19
3,Cobar,4.2,NaN,0.8,NaN,NaN,NaN,NaN,SE,S,...,1022.8,NaN,NaN,8.7,15.6,No,NaN,2023,7,19
4,Launceston,0.7,NaN,1.4,NaN,NaN,NaN,NaN,S,NE,...,NaN,NaN,NaN,3.5,11.2,Yes,NaN,2023,7,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,PearceRAAF,12.4,20.7,0.0,NaN,1.5,NW,57.0,N,NW,...,1012.3,7.0,8.0,14.8,18.9,No,Yes,2023,7,18
94,Dartmoor,5.3,13.7,5.0,NaN,NaN,NW,39.0,NW,SSW,...,1023.3,NaN,NaN,9.3,12.5,Yes,No,2023,7,18
95,GoldCoast,14.8,19.4,3.4,NaN,NaN,SE,30.0,SSE,SSW,...,1020.1,NaN,NaN,17.5,18.5,Yes,No,2023,7,18
96,Bendigo,-1.2,11.9,0.0,NaN,NaN,WNW,31.0,,W,...,1021.9,6.0,8.0,2.2,10.9,No,No,2023,7,18


In [10]:
# générer un rapport d'analyse exploratoire
def generate_eda_report(data, title, output_file):
    profile = ProfileReport(data, title = title, html = {'style':{'full_width':True}})
    profile.to_file(output_file = output_file)


In [11]:
generate_eda_report(df_1, title = "fake report", output_file = "../../reports/eda-report.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]